In [1]:
import numpy as np
from scipy.optimize import minimize
from scipy.special import gammaln, digamma, polygamma
x = np.array([2.4, 3.1, 1.8, 4.5, 2.9, 3.7, 2.2, 5.1, 3.3, 2.8,
              4.0, 3.6, 2.5, 3.9, 4.3, 2.7, 3.4, 4.8, 3.0, 2.6])

n = len(x)
x_bar = np.mean(x)
def neg_log_likelihood(params):
    alpha, beta = params
    
    ll = (n * (alpha * np.log(beta) - gammaln(alpha))
          + (alpha - 1) * np.sum(np.log(x))
          - beta * np.sum(x))
    
    return -ll 
# By Using Newton Raphson Method
A = np.log(x_bar) - np.mean(np.log(x))

# Initial guess (method of moments)
alpha = 1

# Newton–Raphson
change = 1e-4
max_iter = 100

for i in range(max_iter):
    f = np.log(alpha) - digamma(alpha) - A
    f_prime = (1/alpha) - polygamma(1, alpha)
    
    alpha_new = alpha - f / f_prime
    
    if abs(alpha_new - alpha) < change:
        break
        
    alpha = alpha_new

alpha_mle = alpha
theta_mle = alpha_mle / x_bar

print("Newton-Raphson MLEs:")
print("alpha =", alpha_mle)
print("theta =", theta_mle)

# By Using the Built-in Method Minimize
initial_guess = np.array([1.0, 1.0])
result = minimize(neg_log_likelihood, initial_guess, bounds=[(1e-6, None), (1e-6, None)])

alpha_mle, theta_mle = result.x

print("MLE estimates:")
print("alpha =", alpha_mle)
print("theta =", theta_mle)

# Fisher Information Matrix
I11 = n * polygamma(1, alpha_mle)      # trigamma
I12 = - n / theta_mle
I22 = n * alpha_mle / (theta_mle**2)

Fisher = np.array([[I11, I12],
                   [I12, I22]])

print("\nFisher Information Matrix:")
print(Fisher)

cov_matrix = np.linalg.inv(Fisher)
std_errors = np.sqrt(np.diag(cov_matrix))

z = 1.96  # 95% normal quantile

alpha_ci = (alpha_mle - z * std_errors[0],
            alpha_mle + z * std_errors[0])

theta_ci = (theta_mle - z * std_errors[1],
            theta_mle + z * std_errors[1])

print("\n95% Confidence Intervals:")
print("alpha CI:", alpha_ci)
print("theta CI:", theta_ci)


Newton-Raphson MLEs:
alpha = 14.37829871558653
theta = 4.317807422098057
MLE estimates:
alpha = 14.378103364324053
theta = 4.317750534193888

Fisher Information Matrix:
[[ 1.44049671 -4.63204158]
 [-4.63204158 15.42469211]]

95% Confidence Intervals:
alpha CI: (np.float64(5.56790526944431), np.float64(23.188301459203796))
theta CI: (np.float64(1.6253878864260702), np.float64(7.010113181961707))
